# Neural Networks, MLP (perceptrón multicapa)

Dígitos escritos a mano: cuánto acierta una red según el número de neuronas ocultas.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=mlp&tab=realWorld)

Ejecuta la celda de abajo (no hace falta GPU).

In [ ]:
# Dígitos escritos a mano: una red neuronal (MLP) con una capa oculta
import warnings

import numpy as np
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# Cada red entrena 150 épocas y para ahí, aunque podría mejorar un poco más: es a propósito.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

X, y = load_digits(return_X_y=True)  # 1797 imágenes de 8×8 = 64 píxeles
X = X / 16  # cada píxel pasa de 0-16 a 0-1
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(f"{len(X_tr)} imágenes para entrenar y {len(X_te)} nuevas para medir\n")

print("neuronas ocultas | pesos | entrenamiento | datos nuevos")
for n in (2, 8, 32):
    red = MLPClassifier(hidden_layer_sizes=(n,), max_iter=150, random_state=0).fit(X_tr, y_tr)
    pesos = sum(W.size + b.size for W, b in zip(red.coefs_, red.intercepts_))
    print(f"{n:16d} | {pesos:5d} | {red.score(X_tr, y_tr):13.1%} | {red.score(X_te, y_te):12.1%}")

# La red de 32 neuronas entrega una probabilidad por dígito (softmax) para cada imagen nueva
p = red.predict_proba(X_te[:1])[0]
top = np.argsort(p)[::-1][:2]
print(f"\nPrimera imagen nueva (es un {y_te[0]}): {top[0]} con {p[top[0]]:.0%}, {top[1]} con {p[top[1]]:.0%}")

**Opcional: con PyTorch.** La celda siguiente hace lo mismo con PyTorch, la biblioteca que se usa en la práctica. Colab ya lo trae instalado; si no está, la celda solo avisa y no falla. Ejecuta antes la celda del ejercicio.

In [ ]:
# Opcional: una red equivalente en PyTorch (Colab ya lo trae instalado; en el navegador no corre).
# Usa X_tr, X_te, y_tr, y_te de la celda anterior (el ejercicio): ejecútala antes.
# El acierto será parecido al de scikit-learn, no idéntico: cambian el sorteo de los pesos, el orden de los lotes
# y la penalización L2 (scikit-learn usa alpha=0.0001; aquí no hay ninguna).
try:
    import torch
    from torch import nn
except (ImportError, OSError):  # OSError: PyTorch instalado pero roto (falta una biblioteca)
    print("PyTorch no está instalado: esta celda es opcional.")
else:
    torch.manual_seed(0)
    Xt, Xv = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(X_te, dtype=torch.float32)
    yt, yv = torch.tensor(y_tr), torch.tensor(y_te)
    red_pt = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    opt = torch.optim.Adam(red_pt.parameters(), lr=0.001)
    for epoca in range(150):
        orden = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 200):  # lotes de hasta 200 filas, como scikit-learn
            lote = orden[i:i + 200]
            opt.zero_grad()
            perdida = nn.functional.cross_entropy(red_pt(Xt[lote]), yt[lote])
            perdida.backward()
            opt.step()
    acierto = (red_pt(Xv).argmax(dim=1) == yv).float().mean().item()
    print(f"PyTorch, 32 neuronas ocultas: {acierto:.1%} en datos nuevos")